In [1]:
!pip install -q transformers datasets peft accelerate evaluate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/mt5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name,
    torch_dtype="auto"
)

# A100 optimization
model.gradient_checkpointing_enable()
model.config.use_cache = False

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/376 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/560 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [4]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [5]:
from peft import LoraConfig, get_peft_model, TaskType

#r=16
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q", "k", "v", "o"],   # correct for mT5
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 9,437,184 || all params: 1,751,247,872 || trainable%: 0.5389


In [6]:
from datasets import load_from_disk

train_dataset = load_from_disk("/content/drive/MyDrive/byt5_cache/train_dataset")
val_dataset = load_from_disk("/content/drive/MyDrive/byt5_cache/val_dataset")

print("Datasets loaded from cache ✅")

Datasets loaded from cache ✅


In [9]:
def preprocess(example):
    inputs = "fix tamil-english: " + example["generated_script"]
    targets = example["expected_script"]

    model_inputs = tokenizer(
        inputs,
        max_length=256,
        truncation=True
    )

    labels = tokenizer(
        targets,
        max_length=256,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [10]:
train_dataset = train_dataset.map(
    preprocess,
    batched=False,
    num_proc=2
)

val_dataset = val_dataset.map(
    preprocess,
    batched=False,
    num_proc=2
)

Map (num_proc=2):   0%|          | 0/20717 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/5180 [00:00<?, ? examples/s]

In [11]:
train_dataset.save_to_disk("/content/drive/MyDrive/mt5_cache/train_dataset")
val_dataset.save_to_disk("/content/drive/MyDrive/mt5_cache/val_dataset")

print("Tokenized datasets saved ✅")

Saving the dataset (0/1 shards):   0%|          | 0/20717 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5180 [00:00<?, ? examples/s]

Tokenized datasets saved ✅


In [7]:
from datasets import load_from_disk

train_dataset = load_from_disk("/content/drive/MyDrive/mt5_cache/train_dataset")
val_dataset = load_from_disk("/content/drive/MyDrive/mt5_cache/val_dataset")

print("Tokenized datasets loaded ✅")

Tokenized datasets loaded ✅


In [8]:
!pip install evaluate jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 85.9 MB/s eta 0:00:00


In [9]:
import evaluate

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")



In [ ]:
import evaluate
import numpy as np

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    preds = np.argmax(preds, axis=-1)

    # Replace -100
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "wer": wer,
        "cer": cer
    }

In [ ]:
import numpy as np

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # If tuple (sometimes happens)
    if isinstance(preds, tuple):
        preds = preds[0]

    # 🔥 IMPORTANT: DO NOT argmax when using generate
    # preds are already token IDs

    # Replace -100 in labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    # Debug (optional)
    for i in range(min(3, len(decoded_preds))):
        print("PRED:", decoded_preds[i])
        print("LABEL:", decoded_labels[i])
        print("----")

    wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {
        "wer": wer,
        "cer": cer
    }

In [10]:
import numpy as np

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # If tuple (sometimes happens)


    if isinstance(preds, tuple):
        print("preds instances")
        preds = preds[0]

    # ✅ Safety: convert logits → token ids if needed
    if preds.ndim == 3:
        preds = np.argmax(preds, axis=-1)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]

    print("Pred len:", len(decoded_preds))
    print("Label len:", len(decoded_labels))

    wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels)

    return {"wer": wer, "cer": cer}

In [11]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

In [12]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

In [13]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/mt5_lora_output_32_tagged",

    # 🔥 A100 optimized
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_ratio=0.05,

    # evaluation
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    # 🔥 precision
    bf16=True,
    tf32=True,

    # stability
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,

    report_to="none"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
from transformers import GenerationConfig

model.generation_config = GenerationConfig(
    max_length=128,
    num_beams=4,
    do_sample=False,
    no_repeat_ngram_size=3,
    length_penalty=0.8,
    early_stopping=True
)

In [ ]:
def debug_metrics(eval_preds):
    preds, labels = eval_preds

    print("PRED TYPE:", type(preds))
    print("PRED SHAPE:", getattr(preds, "shape", None))

    print("LABEL SHAPE:", labels.shape)

    return {"debug": 0}

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset.select(range(500)),  # fast eval
    data_collator=data_collator,
    compute_metrics=debug_metrics
)

NameError: name 'debug_metrics' is not defined

In [15]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset.select(range(500)),  # fast eval
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer
)

In [16]:
print(type(model))

<class 'peft.peft_model.PeftModelForSeq2SeqLM'>


In [17]:
print(model.base_model.model)

MT5ForConditionalGeneration(
  (shared): Embedding(250112, 1024)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250112, 1024)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k): l

In [18]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 0}.


Epoch,Training Loss,Validation Loss,Wer,Cer
1,1.392111,0.602621,0.141920,0.041424
2,1.204423,0.528775,0.141191,0.041287
3,1.082496,0.480937,0.139976,0.041185
4,0.985055,0.456841,0.139247,0.041048
5,0.912713,0.456273,0.138032,0.040842


Pred len: 500
Label len: 500
Pred len: 500
Label len: 500
Pred len: 500
Label len: 500
Pred len: 500
Label len: 500
Pred len: 500
Label len: 500


TrainOutput(global_step=6475, training_loss=1.9524389978357264, metrics={'train_runtime': 11845.6867, 'train_samples_per_second': 8.745, 'train_steps_per_second': 0.547, 'total_flos': 1.82211009549312e+16, 'train_loss': 1.9524389978357264, 'epoch': 5.0})

In [19]:
predictions = trainer.predict(val_dataset)

preds = predictions.predictions

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(predictions.label_ids, skip_special_tokens=True)



Pred len: 5180
Label len: 5180


OverflowError: out of range integral type conversion attempted

In [20]:
predictions = trainer.predict(val_dataset)

Pred len: 5180
Label len: 5180


In [21]:
import numpy as np

preds = predictions.predictions
labels = predictions.label_ids

# Handle tuple case
if isinstance(preds, tuple):
    preds = preds[0]

# Convert logits → token ids
if preds.ndim == 3:
    preds = np.argmax(preds, axis=-1)

# Replace -100 in labels
labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

# Decode
decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

# Strip spaces
decoded_preds = [p.strip() for p in decoded_preds]
decoded_labels = [l.strip() for l in decoded_labels]

In [22]:
final_wer = wer_metric.compute(
    predictions=decoded_preds,
    references=decoded_labels
)

final_cer = cer_metric.compute(
    predictions=decoded_preds,
    references=decoded_labels
)

print("Final WER:", final_wer)
print("Final CER:", final_cer)

Final WER: 0.13570697574791157
Final CER: 0.04223198993791479


In [23]:
import os

os.makedirs("/content/drive/MyDrive/mt5_results", exist_ok=True)

In [24]:
import pandas as pd

df = pd.DataFrame({
    "input": val_dataset["generated_script"],
    "reference": decoded_labels,
    "prediction": decoded_preds
})

csv_path = "/content/drive/MyDrive/mt5_results/val_predictions.csv"
df.to_csv(csv_path, index=False)

print("Saved CSV at:", csv_path)

Saved CSV at: /content/drive/MyDrive/mt5_results/val_predictions.csv


In [25]:
import pandas as pd
output_df = pd.DataFrame({
    "expected_text": decoded_labels,
    "byt5_output": decoded_preds
})

save_path = "/content/drive/MyDrive/byt5_validation_outputs.csv"
output_df.to_csv(save_path, index=False)

print("Saved to:", save_path)

wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels)
print("Final WER:", wer)

Saved to: /content/drive/MyDrive/byt5_validation_outputs.csv
Final WER: 0.13570697574791157


In [26]:
model.save_pretrained("/content/mt5_lora_final")
tokenizer.save_pretrained("/content/mt5_lora_final")

('/content/mt5_lora_final/tokenizer_config.json',
 '/content/mt5_lora_final/tokenizer.json')

In [27]:
merged_model = model.merge_and_unload()

merged_model.save_pretrained("/content/drive/MyDrive/mt5_merged_model")
tokenizer.save_pretrained("/content/drive/MyDrive/mt5_merged_model")

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:683: UserWarning: Input and output embeddings are no longer tied after merging. Setting `tie_word_embeddings=False` in the model config.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/mt5_merged_model/tokenizer_config.json',
 '/content/drive/MyDrive/mt5_merged_model/tokenizer.json')